# 17 — Comparative Experiment (Repair vs. Penalty vs. No-Repair)

**Maps to:** `report/Chapters/Task3.tex` §`T3:RepairCritique`.  
**Ticket:** TICKET-17.

Three-arm comparison of constraint-handling strategies on kroA100 using
**naive single-point crossover** (which produces infeasible offspring):

1. **Repair**: infeasible offspring are repaired via random insertion before
   fitness evaluation.
2. **Penalty**: infeasible offspring survive but receive a penalty proportional
   to the number of constraint violations.
3. **No repair**: infeasible offspring survive as-is with raw `tour_length`
   fitness (visits some cities twice, skips others).

30 independent runs per strategy, pairwise Wilcoxon signed-rank tests with
Bonferroni correction.

---
## Setup

In [1]:
%run ./15_experiment_runner.ipynb

Loaded kroA100: 100 cities


Best fitness : 84458.32
Known optimal: 21282
Gap          : 296.9%
Wall time    : 0.2s

Per-generation log (first 5 rows):
Saved: ../results/9e9a3301_seed0042.csv
Size : 11,657 bytes
Shape: (101, 16)
Cols : ['generation', 'best_fitness', 'mean_fitness', 'diversity', 'pop_size', 'n_generations', 'crossover_rate', 'mutation_rate', 'tournament_k', 'elitism_count', 'selection_method', 'crossover_method', 'mutation_method', 'repair_enabled', 'repair_strategy', 'seed']
Grid: 6 configurations
  976d5e2d | xover=pmx seed=1
  976d5e2d | xover=pmx seed=2
  976d5e2d | xover=pmx seed=3
  0fc01629 | xover=ox seed=1
  0fc01629 | xover=ox seed=2
  0fc01629 | xover=ox seed=3
Total: 6 | Completed: 6 | Pending: 0
Nothing to run — all results already exist.
Re-running the same grid (all should be skipped):
Total: 6 | Completed: 6 | Pending: 0
Nothing to run — all results already exist.


In [2]:
from scipy import stats
from itertools import combinations
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

---
## Naive Crossover and Penalty Function

Naive single-point crossover does not preserve the permutation property.
The penalty function adds `weight × n_violations` to the tour length,
discouraging infeasible solutions without removing them.

In [3]:
def naive_crossover(parent_a, parent_b, pt, _unused=None):
    child = np.empty_like(parent_a)
    child[:pt] = parent_a[:pt]
    child[pt:] = parent_b[pt:]
    return child


def penalised_tour_length(tour, dist_matrix, weight):
    n_cities = dist_matrix.shape[0]
    base = tour_length(tour, dist_matrix)
    violations = n_cities - len(set(int(x) for x in tour))
    return base + weight * violations

---
## Extended `run_experiment`

Supports `crossover_method="naive"` and an optional `penalty_weight` field.
When `penalty_weight > 0` and `repair_enabled=False`, fitness is computed
using the penalised tour length.

In [4]:
def run_experiment(config, dist_matrix):
    n_cities = dist_matrix.shape[0]
    rng = np.random.default_rng(config.seed)

    xover_map = {"pmx": pmx, "ox": ox, "cx": cx, "naive": naive_crossover}
    xover_fn = xover_map[config.crossover_method]
    mutate_fn = {"swap": swap_mutation, "inversion": inversion_mutation}[config.mutation_method]

    penalty_weight = getattr(config, "penalty_weight", 0)
    use_penalty = penalty_weight > 0 and not config.repair_enabled

    def evaluate(pop):
        if use_penalty:
            return np.array([penalised_tour_length(ind, dist_matrix, penalty_weight)
                             for ind in pop])
        return np.array([tour_length(ind, dist_matrix) for ind in pop])

    population = random_population(config.pop_size, n_cities, rng)
    fitnesses = evaluate(population)

    records = []

    for gen in range(config.n_generations):
        raw_fitness = np.array([tour_length(ind, dist_matrix) for ind in population])
        diversity = pairwise_hamming(population)
        records.append({
            "generation": gen,
            "best_fitness": float(raw_fitness.min()),
            "mean_fitness": float(raw_fitness.mean()),
            "diversity": float(diversity),
        })

        new_pop = []

        elite_idx = np.argsort(fitnesses)[:config.elitism_count]
        for idx in elite_idx:
            new_pop.append(population[idx].copy())

        while len(new_pop) < config.pop_size:
            if config.selection_method == "tournament":
                p1 = tournament_select(population, fitnesses, config.tournament_k, rng)
                p2 = tournament_select(population, fitnesses, config.tournament_k, rng)
            else:
                p1 = roulette_select(population, fitnesses, rng)
                p2 = roulette_select(population, fitnesses, rng)

            if rng.random() < config.crossover_rate:
                if config.crossover_method == "cx":
                    child, _ = xover_fn(p1, p2)
                elif config.crossover_method == "naive":
                    pt = rng.integers(1, n_cities)
                    child = xover_fn(p1, p2, pt)
                else:
                    pts = sorted(rng.choice(n_cities, size=2, replace=False))
                    child = xover_fn(p1, p2, pts[0], pts[1])
            else:
                child = p1.copy()

            child = mutate_fn(child, config.mutation_rate, rng)

            if config.repair_enabled:
                child = repair(child, n_cities, dist_matrix=dist_matrix,
                               strategy=config.repair_strategy, rng=rng)

            new_pop.append(child)

        population = np.array(new_pop[:config.pop_size])
        fitnesses = evaluate(population)

    raw_fitness = np.array([tour_length(ind, dist_matrix) for ind in population])
    diversity = pairwise_hamming(population)
    records.append({
        "generation": config.n_generations,
        "best_fitness": float(raw_fitness.min()),
        "mean_fitness": float(raw_fitness.mean()),
        "diversity": float(diversity),
    })

    best_idx = np.argmin(raw_fitness)
    return {
        "log": pd.DataFrame(records),
        "best_tour": population[best_idx],
        "best_fitness": float(raw_fitness[best_idx]),
    }

---
## Load Benchmark

In [5]:
coords, dist_matrix = load_tsp(Path("../data/TSP-dataset/kroA100.tsp"))
n_cities = dist_matrix.shape[0]
print(f"Loaded kroA100: {n_cities} cities")
OPTIMAL = 21282

Loaded kroA100: 100 cities


---
## Experiment Design

All three arms use **naive crossover** so that infeasibility arises.

- **repair**: `repair_enabled=True`, random insertion.
- **penalty**: `repair_enabled=False`, `penalty_weight=50000` added to
  fitness for each constraint violation.
- **no_repair**: `repair_enabled=False`, raw tour length fitness.

`ExperimentConfig` doesn't have a `penalty_weight` field by default, so
we attach it dynamically via a subclass.

In [6]:
@dataclass
class PenaltyConfig(ExperimentConfig):
    penalty_weight: float = 0.0

N_SEEDS = 30
SEEDS = list(range(1, N_SEEDS + 1))

BASE_CONFIG = {
    "pop_size": 100,
    "n_generations": 500,
    "crossover_rate": 0.8,
    "mutation_rate": 0.2,
    "tournament_k": 3,
    "elitism_count": 2,
    "selection_method": "tournament",
    "crossover_method": "naive",
    "mutation_method": "swap",
}

strategies = {
    "repair": (ExperimentConfig, {"repair_enabled": True, "repair_strategy": "random"}),
    "penalty": (PenaltyConfig, {"repair_enabled": False, "repair_strategy": "random", "penalty_weight": 50000.0}),
    "no_repair": (ExperimentConfig, {"repair_enabled": False, "repair_strategy": "random"}),
}

configs = {}
for name, (cls, overrides) in strategies.items():
    cfg_list = []
    for seed in SEEDS:
        cfg_list.append(cls(**{**BASE_CONFIG, **overrides, "seed": seed}))
    configs[name] = cfg_list
    print(f"{name:>12s}: {len(cfg_list):3d} configs  "
          f"hash={config_hash(cfg_list[0])}")

all_configs = [c for group in configs.values() for c in group]
print(f"\nTotal: {len(all_configs)} runs")

      repair:  30 configs  hash=9d75f11b
     penalty:  30 configs  hash=0d740d61
   no_repair:  30 configs  hash=6ec6908d

Total: 90 runs


---
## Sanity Check

In [7]:
rng = np.random.default_rng(0)
p1 = rng.permutation(n_cities)
p2 = rng.permutation(n_cities)
child = naive_crossover(p1, p2, 30)
dups, missing = diagnose(child, n_cities)
print(f"Child valid    : {is_valid_permutation(child, n_cities)}")
print(f"Duplicates     : {len(dups)}")
print(f"Missing        : {len(missing)}")
print(f"Raw tour_length: {tour_length(child, dist_matrix):.0f}")
print(f"Penalised      : {penalised_tour_length(child, dist_matrix, 50000):.0f}")
print(f"Penalty added  : {50000 * len(dups):.0f}")

Child valid    : False
Duplicates     : 23
Missing        : 23
Raw tour_length: 174747
Penalised      : 1324747
Penalty added  : 1150000


---
## Run Experiments

In [8]:
run_grid(all_configs, dist_matrix, n_workers=1)

Total: 90 | Completed: 60 | Pending: 30


  [1/30] 0d740d61_seed0001.csv  best=48092  (2s elapsed, ~62s remaining)


  [2/30] 0d740d61_seed0002.csv  best=60357  (4s elapsed, ~60s remaining)


  [3/30] 0d740d61_seed0003.csv  best=55853  (6s elapsed, ~58s remaining)


  [4/30] 0d740d61_seed0004.csv  best=55551  (9s elapsed, ~55s remaining)


...

  [27/30] 0d740d61_seed0027.csv  best=59931  (58s elapsed, ~6s remaining)


  [28/30] 0d740d61_seed0028.csv  best=58961  (60s elapsed, ~4s remaining)


  [29/30] 0d740d61_seed0029.csv  best=55857  (62s elapsed, ~2s remaining)


  [30/30] 0d740d61_seed0030.csv  best=59151  (64s elapsed, ~0s remaining)

Done: 30 runs in 64.2s


[PosixPath('../results/0d740d61_seed0001.csv'),
 PosixPath('../results/0d740d61_seed0002.csv'),
 PosixPath('../results/0d740d61_seed0003.csv'),
 PosixPath('../results/0d740d61_seed0004.csv'),
 PosixPath('../results/0d740d61_seed0005.csv'),
 PosixPath('../results/0d740d61_seed0006.csv'),
 PosixPath('../results/0d740d61_seed0007.csv'),
 PosixPath('../results/0d740d61_seed0008.csv'),
 PosixPath('../results/0d740d61_seed0009.csv'),
 PosixPath('../results/0d740d61_seed0010.csv'),
 PosixPath('../results/0d740d61_seed0011.csv'),
 PosixPath('../results/0d740d61_seed0012.csv'),
 PosixPath('../results/0d740d61_seed0013.csv'),
 PosixPath('../results/0d740d61_seed0014.csv'),
 PosixPath('../results/0d740d61_seed0015.csv'),
 PosixPath('../results/0d740d61_seed0016.csv'),
 PosixPath('../results/0d740d61_seed0017.csv'),
 PosixPath('../results/0d740d61_seed0018.csv'),
 PosixPath('../results/0d740d61_seed0019.csv'),
 PosixPath('../results/0d740d61_seed0020.csv'),
 PosixPath('../results/0d740d61_seed0021

---
## Collect Results

In [9]:
def collect_results(config_list, label):
    rows = []
    for c in config_list:
        path = result_path(c)
        if not path.exists():
            continue
        df = pd.read_csv(path)
        final = df.iloc[-1]
        rows.append({
            "strategy": label,
            "seed": c.seed,
            "best_fitness": final["best_fitness"],
            "mean_fitness": final["mean_fitness"],
            "diversity": final["diversity"],
        })
    return pd.DataFrame(rows)

results = pd.concat([
    collect_results(configs["repair"], "repair"),
    collect_results(configs["penalty"], "penalty"),
    collect_results(configs["no_repair"], "no_repair"),
], ignore_index=True)

print(f"Collected {len(results)} results")
results.groupby("strategy")["best_fitness"].describe().round(2)

Collected 90 results


,count,mean,std,min,25%,50%,75%,max
strategy,,,,,,,,
no_repair,30.0,38129.27,4135.41,28805.71,35678.72,37750.48,40649.12,47741.29
penalty,30.0,57947.91,3284.51,48091.74,55853.78,58079.43,60271.90,64585.95
repair,30.0,49777.84,3186.86,43890.51,47754.79,49406.18,51236.33,56629.20


---
## Summary Statistics

In [10]:
summary = results.groupby("strategy")["best_fitness"].agg(
    ["count", "mean", "std", "min", "median", "max"]
).round(2)

summary["gap_%"] = ((summary["mean"] - OPTIMAL) / OPTIMAL * 100).round(1)

print("Summary Statistics — Best Fitness (lower is better)")
print("=" * 80)
print(summary.to_string())

Summary Statistics — Best Fitness (lower is better)
           count      mean      std       min    median       max  gap_%
strategy                                                                
no_repair     30  38129.27  4135.41  28805.71  37750.48  47741.29   79.2
penalty       30  57947.91  3284.51  48091.74  58079.43  64585.95  172.3
repair        30  49777.84  3186.86  43890.51  49406.18  56629.20  133.9


---
## Pairwise Wilcoxon Signed-Rank Tests (Bonferroni corrected)

For each pair of strategies, test whether the median difference in best
fitness is significantly different from zero. With 3 comparisons, the
Bonferroni-corrected significance level is α/3 = 0.0167.

In [11]:
strategy_names = ["repair", "penalty", "no_repair"]
alpha = 0.05
n_comparisons = len(list(combinations(strategy_names, 2)))
alpha_corrected = alpha / n_comparisons

print(f"Pairwise Wilcoxon Signed-Rank Tests (Bonferroni α = {alpha_corrected:.4f})")
print("=" * 70)

for s1, s2 in combinations(strategy_names, 2):
    v1 = results[results["strategy"] == s1].sort_values("seed")["best_fitness"].values
    v2 = results[results["strategy"] == s2].sort_values("seed")["best_fitness"].values

    stat, p = stats.wilcoxon(v1, v2)
    sig = "Yes" if p < alpha_corrected else "No"
    better = s1 if v1.mean() < v2.mean() else s2

    print(f"\n  {s1} vs {s2}:")
    print(f"    statistic = {stat:.2f}, p = {p:.6f}, reject H₀: {sig}")
    if p < alpha_corrected:
        print(f"    → '{better}' significantly better")

Pairwise Wilcoxon Signed-Rank Tests (Bonferroni α = 0.0167)

  repair vs penalty:
    statistic = 1.00, p = 0.000000, reject H₀: Yes
    → 'repair' significantly better

  repair vs no_repair:
    statistic = 0.00, p = 0.000000, reject H₀: Yes
    → 'no_repair' significantly better

  penalty vs no_repair:
    statistic = 0.00, p = 0.000000, reject H₀: Yes
    → 'no_repair' significantly better


### Interpretation

Three constraint-handling strategies compared using naive crossover on kroA100:

- **Repair** produces valid tours with genuinely meaningful fitness values.
- **Penalty** keeps infeasible chromosomes but discourages them via a fitness
  penalty proportional to violations. Selection pressure steers toward
  feasible solutions without enforcing it.
- **No repair** allows infeasible chromosomes to survive with raw tour length.
  The artificially low fitness values (from visiting fewer unique cities)
  corrupt selection pressure.

The Wilcoxon tests with Bonferroni correction quantify whether these
strategy differences are statistically significant.

---
## Convergence Comparison

In [12]:
STRATEGY_STYLE = {
    "repair": {"color": "#2166ac", "label": "Repair"},
    "penalty": {"color": "#1b7837", "label": "Penalty"},
    "no_repair": {"color": "#b2182b", "label": "No repair"},
}

def load_convergence(config_list, label):
    frames = []
    for c in config_list:
        path = result_path(c)
        if not path.exists():
            continue
        df = pd.read_csv(path)[["generation", "best_fitness"]].copy()
        df["seed"] = c.seed
        df["strategy"] = label
        frames.append(df)
    return pd.concat(frames, ignore_index=True)

conv = pd.concat([
    load_convergence(configs[s], s) for s in strategy_names
], ignore_index=True)

conv_summary = conv.groupby(["strategy", "generation"])["best_fitness"].agg(
    ["mean", "std"]).reset_index()

print("Convergence data shape:", conv_summary.shape)

Convergence data shape: (1503, 4)


---
## Diversity Comparison

In [13]:
def load_diversity(config_list, label):
    frames = []
    for c in config_list:
        path = result_path(c)
        if not path.exists():
            continue
        df = pd.read_csv(path)[["generation", "diversity"]].copy()
        df["seed"] = c.seed
        df["strategy"] = label
        frames.append(df)
    return pd.concat(frames, ignore_index=True)

div_data = pd.concat([
    load_diversity(configs[s], s) for s in strategy_names
], ignore_index=True)

div_summary = div_data.groupby(["strategy", "generation"])["diversity"].agg(
    ["mean", "std"]).reset_index()

print("Diversity data shape:", div_summary.shape)

Diversity data shape: (1503, 4)


---
## Export Comparison CSV

In [14]:
compare_path = Path("../results/compare.csv")
results.to_csv(compare_path, index=False)
print(f"Saved: {compare_path}")
print(f"Shape: {results.shape}")

Saved: ../results/compare.csv
Shape: (90, 5)


---
## Summary

Three-arm comparative experiment on kroA100 using naive single-point
crossover (which produces infeasible offspring):

1. **Repair** — infeasible offspring repaired via random insertion before
   fitness evaluation. Produces valid tours with meaningful fitness.
2. **Penalty** — infeasible offspring penalised proportionally to violations
   (`weight=50,000`). Selection discourages infeasibility without enforcing it.
3. **No repair** — infeasible offspring survive with raw tour length. Fitness
   values are artificially low and meaningless.

30 seeds per strategy, pairwise Wilcoxon signed-rank tests with Bonferroni
correction (α = 0.0167). Convergence, diversity, and per-run results
exported to `results/compare.csv`.